# Cars85 Data Frames (dplyr) — Solution Notebook (R)

**Goal:** Clean, transform, and explore the 1985 Automobile data set to decide which car to add to a collection. Core dplyr verbs + pipe.

**Data source (exact uploaded UCI files):**
- `data/imports-85.data` — raw 205 instances (`?` = missing)
- `data/imports-85.names` — official attribute documentation (symboling = insurance risk −3…+3; normalized-losses = relative average loss payment)
- `data/cars85.csv` — analysis-ready (headers, `?`→NA, hyphens→underscores)

**There is essentially no analytical difference** between using the cleaned CSV and loading the raw `.data` file with the column list from `.names`. Results for filter / arrange / mutate on highway_mpg, engine_size, make and price are identical.

## Analysis Flowchart
![Cars85 dplyr Workflow](cars85_dataframe_flowchart.png)


## 0. Setup — Load Libraries

In [ ]:
library(readr)
library(dplyr)

## 1. Load the Data

In [ ]:
cars <- read_csv("data/cars85.csv")
# Raw UCI alternative (identical content):
# cols <- c("symboling","normalized_losses","make","fuel_type","aspiration",
#   "num_of_doors","body_style","drive_wheels","engine_location",
#   "wheel_base","length","width","height","curb_weight",
#   "engine_type","num_of_cylinders","engine_size","fuel_system",
#   "bore","stroke","compression_ratio","horsepower","peak_rpm",
#   "city_mpg","highway_mpg","price")
# cars <- read_csv("data/imports-85.data", col_names = cols, na = "?")
cars
# Expected: 205 rows × 26 columns

## 2. Inspect

In [ ]:
head(cars)
summary(cars)
# Key: 41 NAs in normalized_losses, 4 in price; highway_mpg 16-54; engine_size 61-326
# Most common makes: toyota 32, nissan 18, mazda 17

## 3. Clean — drop normalized_losses

In [ ]:
cars <- cars %>% select(-normalized_losses)
dim(cars)   # 205 × 25
colnames(cars)

## 4. Column names

In [ ]:
colnames(cars)

## 5. Rename symboling → risk_factor
(From .names: +3 = risky, −3 = probably safe)

In [ ]:
cars <- cars %>% rename(risk_factor = symboling)
colnames(cars)

## 6. MPG threshold + mutate

In [ ]:
mpg_threshold <- 30
cars <- cars %>% mutate(mpg_diff_from_threshold = highway_mpg - mpg_threshold)
cars %>% select(make, highway_mpg, mpg_diff_from_threshold, engine_size, price) %>% head(10)

## 7. Filter mpg_diff > 0

In [ ]:
mpg_exceeds_threshold <- cars %>% filter(mpg_diff_from_threshold > 0)
nrow(mpg_exceeds_threshold)  # ~98
mpg_exceeds_threshold %>% select(make, highway_mpg, mpg_diff_from_threshold, engine_size, price) %>% head(8)

## 8. Arrange desc(mpg_diff)

In [ ]:
mpg_exceeds_threshold <- mpg_exceeds_threshold %>% arrange(desc(mpg_diff_from_threshold))
mpg_exceeds_threshold %>% select(make, highway_mpg, mpg_diff_from_threshold, engine_size, price) %>% head(8)
# Top: Honda 54 (+24), Chevrolet 53 (+23), Nissan 50 (+20), Toyota 47 (+17) ...

## 9. Arrange by engine_size desc

In [ ]:
ordered_by_engine_size <- cars %>% arrange(desc(engine_size))
ordered_by_engine_size %>% select(make, engine_size, highway_mpg, price, risk_factor) %>% head(8)
# Largest: Jaguar / Mercedes-Benz 300+

## 10. Chosen make

In [ ]:
chosen_make <- "toyota"
chosen_make_details <- cars %>% filter(make == chosen_make) %>% arrange(desc(engine_size))
chosen_make_details %>% select(make, engine_size, highway_mpg, city_mpg, price, risk_factor)
# toyota n=32, max eng=171, mean hwy≈32.9; try "honda" for higher efficiency

## 11. Value score (optional)

In [ ]:
cars <- cars %>% mutate(value_score = highway_mpg / (price / 1000))
cars %>% filter(mpg_diff_from_threshold > 0, !is.na(value_score)) %>%
  arrange(desc(value_score)) %>%
  select(make, highway_mpg, price, value_score, engine_size) %>% head(10)

---
## Alternate Code Paths

In [ ]:
# Base R
cars_base <- read_csv("data/cars85.csv")
cars_base$normalized_losses <- NULL
names(cars_base)[names(cars_base)=="symboling"] <- "risk_factor"
mpg_ex_base <- cars_base[cars_base$highway_mpg > 30, ]
mpg_ex_base <- mpg_ex_base[order(-mpg_ex_base$highway_mpg), ]
head(mpg_ex_base[, c("make","highway_mpg","engine_size")])

# transmute slim
slim <- cars %>% transmute(make, risk_factor, highway_mpg, engine_size, price,
                           mpg_diff = highway_mpg - 30,
                           value_score = highway_mpg / (price / 1000))
head(slim)

---
## More Practice

In [ ]:
# Practice 1
affordable <- cars %>% filter(price < 10000, body_style %in% c("sedan","hatchback")) %>% arrange(price)
nrow(affordable); head(affordable %>% select(make, body_style, price, highway_mpg), 6)

# Practice 2
cars <- cars %>% mutate(risk_label = case_when(
  risk_factor <= 0 ~ "safe", risk_factor == 1 ~ "neutral", risk_factor >= 2 ~ "risky"))
table(cars$risk_label)

# Practice 3
bind_rows(
  cars %>% filter(make=="toyota") %>% arrange(desc(highway_mpg)) %>% slice(1),
  cars %>% filter(make=="nissan") %>% arrange(desc(highway_mpg)) %>% slice(1),
  cars %>% filter(make=="mazda")  %>% arrange(desc(highway_mpg)) %>% slice(1)
) %>% select(make, highway_mpg, engine_size, price)

---
## Simulation

In [ ]:
sim_mpg_threshold <- 30; sim_chosen_make <- "honda"
sim_max_price <- 12000; sim_min_engine <- 90
sim_result <- cars %>%
  mutate(mpg_diff = highway_mpg - sim_mpg_threshold) %>%
  filter(mpg_diff > 0, (make == sim_chosen_make | sim_chosen_make == "ANY"),
         (is.na(price) | price <= sim_max_price), engine_size >= sim_min_engine) %>%
  arrange(desc(mpg_diff)) %>%
  select(make, body_style, highway_mpg, mpg_diff, engine_size, price, risk_factor)
cat("Surviving cars:", nrow(sim_result), "
"); print(head(sim_result, 8))

thresholds <- c(20,25,28,30,32,35,40,45,50)
data.frame(threshold = thresholds,
           n_cars = sapply(thresholds, function(th) sum(cars$highway_mpg > th, na.rm=TRUE)))

## Reflection
1. Which car(s) would you collect and why?  
2. Sensitivity to threshold / make?  
3. Extra data still needed (reliability, TCO, current market values)?

You now have a complete dplyr workflow on the authentic UCI 1985 Automobile data set.
